In [75]:
import pandas as pd

movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

print("Movies:")
display(movies.head(10))

print("Ratings:")
display(ratings.head(10))

Movies:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


Ratings:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


In [76]:
print(movies.shape)
print(ratings.shape)

(9742, 3)
(100836, 4)


In [77]:
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)

display(movies.head())

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [78]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['genres'])

print(tfidf_matrix.shape)

(9742, 23)


In [79]:
from sklearn.metrics.pairwise import cosine_similarity

cos_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(cos_sim.shape)

(9742, 9742)


In [80]:
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

print(indices.head(10))

title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
Heat (1995)                           5
Sabrina (1995)                        6
Tom and Huck (1995)                   7
Sudden Death (1995)                   8
GoldenEye (1995)                      9
dtype: int64


In [81]:
def recommend(title, cos_sim=cos_sim):

    idx = indices[title]

    sim_scores = list(enumerate(cos_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:101]

    movie_indices = [i[0] for i in sim_scores]

    scores = [i[1] for i in sim_scores]

    result =  movies[['title', 'genres']].iloc[movie_indices].copy()
    
    result['similarity_score'] = scores 
    
    return result

In [82]:
recommend("Toy Story (1995)")

,title,genres,similarity_score
1706,Antz (1998),Adventure Animation Children Comedy Fantasy,1.000000
2355,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy,1.000000
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy,1.000000
3000,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy,1.000000
3568,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy,1.000000
...,...,...,...
4151,My Neighbor Totoro (Tonari no Totoro) (1988),Animation Children Drama Fantasy,0.835546
12,Balto (1995),Adventure Animation Children,0.833737
1549,101 Dalmatians (One Hundred and One Dalmatians...,Adventure Animation Children,0.833737
1552,"Rescuers Down Under, The (1990)",Adventure Animation Children,0.833737


In [83]:
tags = pd.read_csv("data/tags.csv")

display(tags.head(10))

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
5,2,89774,Tom Hardy,1445715205
6,2,106782,drugs,1445715054
7,2,106782,Leonardo DiCaprio,1445715051
8,2,106782,Martin Scorsese,1445715056
9,7,48516,way too long,1169687325


In [84]:
tags = tags[['movieId', 'tag']]

display(tags.head(10))

,movieId,tag
0,60756,funny
1,60756,Highly quotable
2,60756,will ferrell
3,89774,Boxing story
4,89774,MMA
5,89774,Tom Hardy
6,106782,drugs
7,106782,Leonardo DiCaprio
8,106782,Martin Scorsese
9,48516,way too long


In [85]:
tags_grouped = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x.astype(str)))

display(tags_grouped.head(10))

movieId
1                                  pixar pixar fun
2     fantasy magic board game Robin Williams game
3                                        moldy old
5                                 pregnancy remake
7                                           remake
11                              politics president
14                              politics president
16                                           Mafia
17                                     Jane Austen
21                                       Hollywood
Name: tag, dtype: str

In [86]:
movies = movies.merge(tags_grouped, on='movieId', how='left')

display(movies.head(10))

,movieId,title,genres,tag
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,pixar pixar fun
1,2,Jumanji (1995),Adventure Children Fantasy,fantasy magic board game Robin Williams game
2,3,Grumpier Old Men (1995),Comedy Romance,moldy old
3,4,Waiting to Exhale (1995),Comedy Drama Romance,NaN
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake
5,6,Heat (1995),Action Crime Thriller,NaN
6,7,Sabrina (1995),Comedy Romance,remake
7,8,Tom and Huck (1995),Adventure Children,NaN
8,9,Sudden Death (1995),Action,NaN
9,10,GoldenEye (1995),Action Adventure Thriller,NaN


In [87]:
movies['tag'] = movies['tag'].fillna('')

movies['metadata'] = movies['genres'] + ' ' + movies['tag']

display(movies[['title', 'metadata']].head())

,title,metadata
0,Toy Story (1995),Adventure Animation Children Comedy Fantasy pi...
1,Jumanji (1995),Adventure Children Fantasy fantasy magic board...
2,Grumpier Old Men (1995),Comedy Romance moldy old
3,Waiting to Exhale (1995),Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy pregnancy remake


In [88]:
tfidf_matrix = tfidf.fit_transform(movies['metadata'])

cos_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [145]:
def recommend(title, num=10, cos_sim=cos_sim):

    if title not in indices:
        return "Movie not found."

    idx = indices[title]

    sim_scores = list(enumerate(cos_sim[idx]))

    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:num+1]

    movie_indices = [i[0] for i in sim_scores]

    scores = [i[1] for i in sim_scores]

    result = movies[['title', 'metadata']].iloc[movie_indices].copy()

    result['similarity_score'] = scores

    result['similarity_score'] = result['similarity_score'].round(4)

    result = result.reset_index(drop=True)

    return result

In [148]:
recommend("Toy Story (1995)")

,title,metadata,similarity_score
0,"Bug's Life, A (1998)",Adventure Animation Children Comedy Pixar,0.8622
1,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy an...,0.6440
2,Guardians of the Galaxy 2 (2017),Action Adventure Sci-Fi fun,0.3677
3,Antz (1998),Adventure Animation Children Comedy Fantasy,0.3579
4,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure Animation Children Comedy Fantasy,0.3579
5,"Emperor's New Groove, The (2000)",Adventure Animation Children Comedy Fantasy,0.3579
6,"Monsters, Inc. (2001)",Adventure Animation Children Comedy Fantasy,0.3579
7,"Wild, The (2006)",Adventure Animation Children Comedy Fantasy,0.3579
8,Shrek the Third (2007),Adventure Animation Children Comedy Fantasy,0.3579
9,"Tale of Despereaux, The (2008)",Adventure Animation Children Comedy Fantasy,0.3579
